# 01 Data loading

Loads the balanced dataset (`data/processed/domains_balanced.csv.gz`, built by the scripts in `src/data/`), splits it 80/10/10 into train/validation/test, stratified by family with a fixed seed, and computes the sample weights. The same code (`src/data/dataset.py`) is used by every model, so all models see the same rows.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

In [2]:
import pandas as pd
from src.data.dataset import load_splits

train, val, test = load_splits()
train.head()

,domain,label,family,weight
0,6j29qz11hsefgh7oqsr1vpapri.com,dga,gameover,0.702975
1,qckuygxeyemxgongvylveeoh.cx,dga,necurs,0.702975
2,arenaplus.net,legit,legit,1.000014
3,fd5q27kyqb.com,legit,legit,1.000014
4,eksisozluk1923.com,legit,legit,1.000014


## Split sizes

In [3]:
pd.DataFrame({name: df["label"].value_counts() for name, df in
              [("train", train), ("val", val), ("test", test)]}).T

label,dga,legit
train,799754,799732
val,99970,99966
test,99947,99966


## Families

`legit`, `other_dga` (the 59 families with fewer than 300 domains) and 102 DGA families, each with 300 to 29,120 domains.

In [4]:
fam = train.loc[train["label"] == "dga", "family"].value_counts()
print(f"{fam.size} DGA families in train, {fam.min()} to {fam.max()} domains each")
fam.describe()

103 DGA families in train, 240 to 23296 domains each


count      103.000000
mean      7764.601942
std       9061.546973
min        240.000000
25%        743.500000
50%       3072.000000
75%      14217.000000
max      23296.000000
Name: count, dtype: float64

## Sample weights

Legitimate and DGA weigh half each; within DGA a domain's weight is proportional to 1/sqrt(family size), so small families count more without dominating.

In [5]:
w = train.groupby("label")["weight"].agg(["sum", "min", "max"])
dga_w = train.loc[train["label"] == "dga", "weight"]
print(f"DGA weight max/min: {dga_w.max() / dga_w.min():.1f}")
w

DGA weight max/min: 9.9


,sum,min,max
label,,,
dga,799743.0,0.702975,6.925877
legit,799743.0,1.000014,1.000014
